In [16]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
np.random.seed(42)

In [18]:


control = pd.DataFrame({
    'user_id' : np.arange(1,5001),
    'group'   : 'control',
    # 42% conversion rate — some ordered (1), some didn't (0)
    'converted' : np.random.binomial(1,0.42,5000),
    'revenue' : np.random.normal(350,80,5000)
})

In [19]:
treatment = pd.DataFrame({
    'user_id' : np.arange(5001,10001),
    'group'   : 'treatment',
    # 42% conversion rate — some ordered (1), some didn't (0)
    'converted' : np.random.binomial(1,0.52,5000),
    'revenue' : np.random.normal(390,85,5000)
})

In [20]:
# Combine both groups
df = pd.concat([control,treatment], ignore_index=True)

df.head()

,user_id,group,converted,revenue
0,1,control,0,302.242057
1,2,control,1,158.775651
2,3,control,1,317.022340
3,4,control,1,423.077899
4,5,control,0,393.010392


In [21]:
print(df.groupby('group')['converted'].agg(['sum','count','mean']))

            sum  count    mean
group                         
control    2111   5000  0.4222
treatment  2623   5000  0.5246


**Step 2 — Check Basic Numbers**

In [22]:
# Conversion rate per group
summary  = df.groupby('group').agg(
    total_users=('user_id', 'count'),
    total_orders=('converted', 'sum'),
    conversion_rate=('converted', 'mean'),
    avg_revenue=('revenue', 'mean')
).round(4)

In [24]:
summary

,total_users,total_orders,conversion_rate,avg_revenue
group,,,,
control,5000,2111,0.4222,348.5253
treatment,5000,2623,0.5246,392.3786


In [25]:
summary['conversion_rate'] = summary['conversion_rate'] * 100
print(summary)

           total_users  total_orders  conversion_rate  avg_revenue
group                                                             
control           5000          2111            42.22     348.5253
treatment         5000          2623            52.46     392.3786


In [26]:
# Calculate lift

control_rate = summary.loc['control','conversion_rate']
treatment_rate = summary.loc['treatment','conversion_rate']
lift = treatment_rate - control_rate
print(f'\nlift:{lift}')
print(f"Relative improvement: {(lift/control_rate*100):.1f}%")


lift:10.239999999999995
Relative improvement: 24.3%


In [33]:
#Step 3 — The Most Important Part — Is It Statistically Significant

# Separate the two groups
control_converted = df[df['group']=='control']['converted']
treatment_converted = df[df['group']=='treatment']['converted']


In [34]:
# Chi-square test for conversion rate
# (use this when measuring % conversion — yes/no outcome)
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['group'], df['converted'])
print("Contingency Table:")
print(contingency_table)

Contingency Table:
converted     0     1
group                
control    2889  2111
treatment  2377  2623


In [37]:
#chi2, p_value, dof, expected = chi2_contingency(contingency_table)

chi2, p_value, dof, excepted = chi2_contingency(contingency_table)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")


Chi-square statistic: 104.7449
P-value: 0.000000
Degrees of freedom: 1


In [36]:
# Decision
alpha = 0.05  # 5% significance level
if p_value < alpha:
    print("\n✅ SIGNIFICANT — The discount popup GENUINELY works!")
    print("   We can confidently roll it out to all users.")
else:
    print("\n❌ NOT SIGNIFICANT — Could be random luck.")
    print("   Do not roll out yet.")


✅ SIGNIFICANT — The discount popup GENUINELY works!
   We can confidently roll it out to all users.


Step 4 — Revenue Test (t-test)

In [41]:
# t-test for revenue difference
# (use this when measuring a continuous number like revenue)

control_revenue = df[df['group'] == 'control']['revenue']
treatment_revenue = df[df['group'] == 'treatment']['revenue']

t_stat, p_value_revenue = stats.ttest_ind(control_revenue,treatment_revenue)

print(f"Control avg revenue:   ₹{control_revenue.mean():.2f}")
print(f"Treatment avg revenue: ₹{treatment_revenue.mean():.2f}")
print(f"Difference:            ₹{treatment_revenue.mean()-control_revenue.mean():.2f}")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value_revenue:.6f}")

if p_value_revenue < 0.05:
    print("\n✅ Revenue increase is also statistically significant!")

Control avg revenue:   ₹348.53
Treatment avg revenue: ₹392.38
Difference:            ₹43.85

t-statistic: -26.1629
p-value: 0.000000

✅ Revenue increase is also statistically significant!


Step 5 — Confidence Interval

In [42]:
import scipy.stats as st

# 95% confidence interval for conversion rate difference

n_c = len(control_converted)
n_t = len(treatment_converted)
p_c = control_converted.mean()
p_t = treatment_converted.mean()

# Standard error
se = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)

# 95% CI = difference ± 1.96 * standard error
diff = p_t - p_c
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print(f"Conversion rate difference: {diff*100:.2f}%")
print(f"95% Confidence Interval: ({ci_lower*100:.2f}%, {ci_upper*100:.2f}%)")
print("\nInterpretation:")
print(f"We are 95% confident the true lift is between")
print(f"{ci_lower*100:.1f}% and {ci_upper*100:.1f}%")

Conversion rate difference: 10.24%
95% Confidence Interval: (8.29%, 12.19%)

Interpretation:
We are 95% confident the true lift is between
8.3% and 12.2%


In [44]:
print("=" * 55)
print("   A/B TEST RESULTS — SWIGGY DISCOUNT POPUP")
print("=" * 55)
print(f"\nTest Duration:     2 weeks")
print(f"Sample Size:       5,000 per group")
print(f"\nControl Rate:      {p_c*100:.1f}%")
print(f"Treatment Rate:    {p_t*100:.1f}%")
print(f"Lift:              +{diff*100:.1f}%")
print(f"P-value:           {p_value:.6f} (< 0.05 ✅)")
print(f"95% CI:            ({ci_lower*100:.1f}%, {ci_upper*100:.1f}%)")
print(f"\nRevenue Impact:    +₹{treatment_revenue.mean()-control_revenue.mean():.0f} per user")
print(f"\nRECOMMENDATION:")
print(f"→ Roll out discount popup to ALL users.")
print(f"→ Expected 23.7% improvement in conversion rate.")
print("=" * 55)

   A/B TEST RESULTS — SWIGGY DISCOUNT POPUP

Test Duration:     2 weeks
Sample Size:       5,000 per group

Control Rate:      42.2%
Treatment Rate:    52.5%
Lift:              +10.2%
P-value:           0.000000 (< 0.05 ✅)
95% CI:            (8.3%, 12.2%)

Revenue Impact:    +₹44 per user

RECOMMENDATION:
→ Roll out discount popup to ALL users.
→ Expected 23.7% improvement in conversion rate.
